# SparseLoCo + LoRA vs Baseline — Fine-tuning Distribuído Simulado

Comparação entre duas abordagens de fine-tuning do **Covenant-72B** no benchmark **PubMedQA** (domínio biomédico):

| Abordagem | Parâmetros treináveis | Comunicação |
|---|---|---|
| **Baseline** — AdamW + LoRA centralizado | LoRA adapters | sem compressão |
| **SparseLoCo + LoRA** — distribuído simulado | LoRA adapters | Top-k comprimido |

**Pergunta de pesquisa:** O fine-tuning com SparseLoCo+LoRA atinge accuracy comparável ao LoRA centralizado no PubMedQA? Se sim, o SparseLoCo viabiliza fine-tuning distribuído em domínio específico sem degradar a qualidade do modelo.

> ⚠️ **Requisito de hardware:** Covenant-72B com 4-bit quantization exige ~36GB de VRAM (A100 80GB recomendado). Para rodar em T4 (16GB), substitua `MODEL_ID` por `"meta-llama/Llama-3.2-3B"` na célula de configuração.

In [ ]:
# Instala dependências (execute uma vez)
# !pip install torch transformers peft datasets matplotlib tqdm scikit-learn bitsandbytes accelerate

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset, Subset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Hiperparâmetros

In [ ]:
# ── Modelo ───────────────────────────────────────────────────
# Verifique o model ID correto em huggingface.co/tplr-ai
MODEL_ID = "tplr-ai/covenant-72b"
# Fallback para hardware limitado (T4/16GB):
# MODEL_ID = "meta-llama/Llama-3.2-3B"

# ── Distribuição ─────────────────────────────────────────────
R        = 4      # workers simulados
H        = 30     # inner steps por worker por round
T        = 20     # outer rounds

# ── Modelo e dados ───────────────────────────────────────────
B        = 4      # batch size
MAX_LEN  = 256    # aumentado para caber pergunta + abstract biomédico
N_LABELS = 3      # yes / no / maybe

# ── Otimizadores ─────────────────────────────────────────────
LR_INNER = 3e-4   # AdamW inner lr
LR_OUTER = 1.0    # outer lr α
BETA     = 0.95   # decay do error feedback
TOPK_K   = 0.10   # fração dos params LoRA mantidos (Top-k)

LABEL_MAP = {"yes": 0, "no": 1, "maybe": 2}

SEED = 42
torch.manual_seed(SEED)

## 2. Dataset — PubMedQA (Domínio Biomédico)

PubMedQA é um benchmark de Q&A biomédica baseado em abstracts do PubMed. Dado um abstract científico e uma pergunta, o modelo deve responder **yes / no / maybe** (3 classes).

É um domínio claramente distinto do pré-treinamento do Covenant-72B (web text, código, matemática), tornando o fine-tuning genuinamente necessário para boa performance.

- **1.000 amostras** rotuladas — 800 treino / 200 validação
- Input: pergunta + contexto do abstract (truncado em 256 tokens)
- Labels: yes → 0, no → 1, maybe → 2

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

raw = load_dataset("pubmed_qa", "pqa_labeled", split="train")

class PubMedQADataset(Dataset):
    def __init__(self, data):
        self.samples = data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        item = self.samples[i]
        # Combina pergunta com o abstract (contextos concatenados)
        context = " ".join(item["context"]["contexts"])
        text = item["question"] + " " + context
        enc = tok(text, truncation=True, max_length=MAX_LEN,
                  padding="max_length", return_tensors="pt")
        label = LABEL_MAP[item["final_decision"]]
        return enc["input_ids"].squeeze(), enc["attention_mask"].squeeze(), \
               torch.tensor(label)

# Split manual: 800 treino / 200 validação
raw = raw.shuffle(seed=SEED)
train_data = raw.select(range(800))
val_data   = raw.select(range(800, 1000))

train_ds = PubMedQADataset(train_data)
val_ds   = PubMedQADataset(val_data)

per = len(train_ds) // R
shards = [
    DataLoader(Subset(train_ds, range(r*per, (r+1)*per)),
               batch_size=B, shuffle=True, drop_last=True)
    for r in range(R)
]
central_loader = DataLoader(train_ds, batch_size=B, shuffle=True, drop_last=True)
val_loader     = DataLoader(val_ds,   batch_size=B, shuffle=False)

print(f"Treino: {len(train_ds)} amostras | {R} workers × {per} cada")
print(f"Validação: {len(val_ds)} amostras")
print(f"Classes: {LABEL_MAP}")

## 3. Modelo — Covenant-72B + QLoRA para Classificação

Covenant-72B carregado com **4-bit quantization (QLoRA)** para caber em GPU com memória limitada. LoRA rank=8 aplicado nas camadas de atenção e MLP — apenas uma fração pequena dos parâmetros é treinável.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def make_model():
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=N_LABELS,
        quantization_config=bnb_config,
        device_map="auto",
    )
    base.config.pad_token_id = tok.pad_token_id
    base = prepare_model_for_kbit_training(base)

    lora = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.0,
        bias="none",
    )
    return get_peft_model(base, lora)

def lora_params(m):
    return [p for p in m.parameters() if p.requires_grad]

def to_vec(params):
    return torch.cat([p.detach().cpu().float().flatten() for p in params])

def load_vec(v, params):
    off = 0
    for p in params:
        n = p.numel()
        p.data.copy_(v[off:off+n].view(p.shape).to(p.device).to(p.dtype))
        off += n

m_test = make_model()
m_test.print_trainable_parameters()

## 4. Funções de Treino e Avaliação

In [ ]:
def step_loss(m, ids, mask, labels):
    out = m(input_ids=ids.to(DEVICE),
            attention_mask=mask.to(DEVICE),
            labels=labels.to(DEVICE))
    return out.loss

@torch.no_grad()
def evaluate(m, loader):
    m.eval()
    preds, targets = [], []
    for ids, mask, labels in loader:
        out = m(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE))
        preds.extend(out.logits.argmax(-1).cpu().tolist())
        targets.extend(labels.tolist())
    m.train()
    return accuracy_score(targets, preds)

def topk_sparsify(v, k):
    out = torch.zeros_like(v)
    idx = v.abs().topk(k).indices
    out[idx] = v[idx]
    return out

## 5. Treinamento — SparseLoCo + LoRA

Cada round:
1. R workers treinam H steps localmente a partir dos parâmetros globais
2. Pseudo-gradiente: Δᵣ = θ_global − θ_local
3. Error Feedback + Top-k compressão
4. Outer update: θ_global ← θ_global − α · média(Δ̂ᵣ)

In [ ]:
def run_sparseloco(m, shards, val_loader):
    ps = lora_params(m)
    d  = sum(p.numel() for p in ps)
    k  = max(1, int(d * TOPK_K))
    ef = [torch.zeros(d) for _ in range(R)]

    opts       = [torch.optim.AdamW(ps, lr=LR_INNER) for _ in range(R)]
    opt_states = [None] * R

    print(f'LoRA params: {d:,} | Top-k: k={k} ({TOPK_K*100:.0f}%)')
    history = []

    for t in tqdm(range(T), desc='SparseLoCo'):
        g_vec      = to_vec(ps)
        local_vecs = []
        round_loss = 0.0

        for r in range(R):
            load_vec(g_vec.clone(), ps)
            if opt_states[r] is not None:
                opts[r].load_state_dict(opt_states[r])
            it = iter(shards[r])
            for _ in range(H):
                try:    ids, mask, labels = next(it)
                except: it = iter(shards[r]); ids, mask, labels = next(it)
                opts[r].zero_grad()
                loss = step_loss(m, ids, mask, labels)
                loss.backward()
                opts[r].step()
                round_loss += loss.item()
            opt_states[r] = opts[r].state_dict()
            local_vecs.append(to_vec(ps))

        agg = torch.zeros(d)
        for r, lv in enumerate(local_vecs):
            pg     = g_vec - lv
            ef[r]  = BETA * ef[r] + pg
            comp   = topk_sparsify(ef[r], k)
            ef[r] -= comp
            agg   += comp
        agg /= R

        load_vec(g_vec - LR_OUTER * agg, ps)

        acc = evaluate(m, val_loader)
        history.append({'train_loss': round_loss / (R*H), 'val_acc': acc})
        tqdm.write(f'  round={t:2d}  loss={history[-1]["train_loss"]:.4f}  val_acc={acc:.4f}')

    return history

## 6. Treinamento — Baseline (AdamW + LoRA centralizado)

In [ ]:
def run_baseline(m, loader, val_loader):
    ps  = lora_params(m)
    opt = torch.optim.AdamW(ps, lr=LR_INNER)
    it  = iter(loader)
    steps_per_round = R * H
    history = []

    for t in tqdm(range(T), desc='Baseline'):
        acc_loss = 0.0
        for _ in range(steps_per_round):
            try:    ids, mask, labels = next(it)
            except: it = iter(loader); ids, mask, labels = next(it)
            opt.zero_grad()
            loss = step_loss(m, ids, mask, labels)
            loss.backward()
            opt.step()
            acc_loss += loss.item()

        acc = evaluate(m, val_loader)
        history.append({'train_loss': acc_loss / steps_per_round, 'val_acc': acc})
        tqdm.write(f'  round={t:2d}  loss={history[-1]["train_loss"]:.4f}  val_acc={acc:.4f}')

    return history

## 7. Executar Experimentos

In [ ]:
print('=== SparseLoCo + LoRA ===')
m_sloco = make_model()
hist_sloco = run_sparseloco(m_sloco, shards, val_loader)

print('\n=== Baseline: AdamW + LoRA ===')
m_base = make_model()
hist_base = run_baseline(m_base, central_loader, val_loader)

## 8. Resultados

In [ ]:
sloco_loss = [h['train_loss'] for h in hist_sloco]
sloco_acc  = [h['val_acc']   for h in hist_sloco]
base_loss  = [h['train_loss'] for h in hist_base]
base_acc   = [h['val_acc']   for h in hist_base]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(sloco_loss, 'o-', ms=4, label=f'SparseLoCo+LoRA (Top-{TOPK_K*100:.0f}%)')
ax1.plot(base_loss,  's-', ms=4, label='AdamW+LoRA (baseline)')
ax1.set_xlabel('Outer round'); ax1.set_ylabel('Training loss')
ax1.set_title('Training Loss'); ax1.legend()

ax2.plot(sloco_acc, 'o-', ms=4, label=f'SparseLoCo+LoRA (Top-{TOPK_K*100:.0f}%)')
ax2.plot(base_acc,  's-', ms=4, label='AdamW+LoRA (baseline)')
ax2.set_xlabel('Outer round'); ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy — PubMedQA (yes/no/maybe)'); ax2.legend()
ax2.set_ylim(0.2, 1.0)
ax2.axhline(y=1/3, color='gray', linestyle='--', alpha=0.5, label='chance (33%)')

plt.suptitle(f'SparseLoCo+LoRA vs AdamW+LoRA — Covenant-72B | PubMedQA  (R={R}, H={H}, β={BETA})')
plt.tight_layout()
plt.savefig('resultados.png', dpi=150)
plt.show()

print(f'\nVal Accuracy final  SparseLoCo: {sloco_acc[-1]:.4f}')
print(f'Val Accuracy final  Baseline:   {base_acc[-1]:.4f}')
print(f'Chance level (random):          {1/N_LABELS:.4f}')